# RetinAnalysis tutorial: from a recording to results

This notebook walks one real experiment from raw database entries to the numbers we report, and stops at each point where **we made an analysis choice**: what the choice is, what we picked, why, and how to check it on your own data. The numbered demos after this assume you've been through it, so they only have one line per cell.

**Before running:**
- Docker Desktop is open and the `retinanalysis-database` container is running (see the README).
- The notebook kernel (top right) is `retinanalysis`.
- Run top to bottom. Each section only needs the ones before it.

**Sections:** 0 setup · 1 the pieces (experiments, datafiles, chunks, pipelines) · 2 EI matching · 3 responses and normalization · 4 receptive fields and direction/orientation tuning · 5 saving figures and data

## 0. Setup

In [ ]:
import retinanalysis as ra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Temporary: fixes garbage timecourses on Windows (visionloader double-array bug). Must run before any pipeline is built.
ra.patch_vision_double_array_bug()

## 1. The pieces

**How a recording is organized.** In Symphony, one retina (an *experiment*, e.g. `20251016A`) is recorded in *groups*, usually one per light level (NDF 5 → NDF 0). Each protocol run inside a group is saved as its own *datafile* (`data016`, `data017`, …). Spike sorting (Kilosort) is done per *chunk* of datafiles, so the same physical cell can get **different cell IDs in different chunks**.

**What each piece gives you:**
- The **SpatialNoise (white noise) run** gives STAs, i.e. receptive fields, and it's where cell types are assigned: the `.classificationXX.txt` files (`YT`, `JK`, `ag`, … = whoever classified). In code this is the `AnalysisChunk`.
- A **protocol run** (gratings, flashes, …) gives the responses you want to analyze: `StimBlock` (what was shown) + `ResponseBlock` (spikes).
- A **pipeline** (`ra.create_mea_pipeline`) bundles one protocol datafile with a reference noise chunk and links their cell IDs by EI matching (section 2), so every responding cell gets a cell type and an RF.

This tutorial uses `20251016A`, a C57 wild-type retina that has SpatialNoise, ContrastResponseGrating, ContrastResponseFlash and GratingDSOS all at NDF 0. If it isn't in your database, `20251222A` (used by the contrast demos) works for sections 1–3 and the RF part of 4.

In [ ]:
EXP_NAME = '20251016A'

noise_search = ra.get_datasets_from_protocol_names('manookinlab.protocols.SpatialNoise', b_exact_match=True, verbose=False)
contrast_search = ra.get_datasets_from_protocol_names('contrast', verbose=False)
dsos_search = ra.get_datasets_from_protocol_names('manookinlab.protocols.GratingDSOS', b_exact_match=True, verbose=False)

this_exp = pd.concat([noise_search, contrast_search, dsos_search]).query('exp_name == @EXP_NAME')
if len(this_exp) == 0:
    raise ValueError(f'{EXP_NAME} is not in your database -- run ra.populate_database() or pick another exp_name.')
display(this_exp[['datafile_name', 'NDF', 'protocol_name', 'chunk_name']].drop_duplicates().sort_values('datafile_name'))

**Decision: which noise chunk and classification file to use for cell types.** `ra.find_classified_noise_chunk` prefers a classified chunk at NDF 0, then NDF 1, and prints what it found. If a chunk has several classification files, the first one found is used unless you name one. Cell-type counts can differ between classifiers, so when comparing experiments, use the same classifier's files throughout and say which in the methods.

In [ ]:
ANALYSIS_CHUNK = ra.find_classified_noise_chunk(EXP_NAME)   # or set by hand, e.g. 'data018'
TYPING_FILE = None                                            # e.g. 'data018.classificationYT.txt'; None = first found
print('Reference (typing) chunk:', ANALYSIS_CHUNK)

## 2. EI matching: linking the same cell across runs

**Why it's needed.** Because each chunk is sorted separately, cell 212 in the grating run isn't necessarily cell 212 in the noise run. To give a grating-responding cell its cell type and RF, we find its partner in the noise chunk.

**How.** Each cell has an *electrical image* (EI): its average spike waveform on every electrode, a fingerprint of where it sits on the array. `ra.cluster_match` correlates every protocol cell's EI with every noise cell's EI three ways (full waveform, spatial footprint, power per electrode). It accepts a pair only if:
- they're each other's best match,
- the best match clearly beats the second best (the runner-up must be under 90% of it),
- the correlation clears `corr_cutoff`.

Cells that fail any of these are left out as `Unmatched`, not guessed.

**Decision: `corr_cutoff = 0.85`.** That's the lab convention, carried over from the MATLAB code. Note that `ra.create_mea_pipeline` defaults to **0.8** if you don't pass it, so always pass it explicitly. Higher = fewer but safer matches.

In [ ]:
CORR_CUTOFF = 0.85
GRATING_PROTOCOL = 'manookinlab.protocols.ContrastResponseGrating'

grating_datafile = ra.get_ndf_blocks_for_protocol(EXP_NAME, GRATING_PROTOCOL, verbose=False).query('NDF == 0')['datafile_name'].iloc[0]
with ra.scrollable_prints():
    pipeline = ra.create_mea_pipeline(EXP_NAME, grating_datafile, analysis_chunk_name=ANALYSIS_CHUNK,
                                      typing_file=TYPING_FILE, corr_cutoff=CORR_CUTOFF)

n_protocol, n_noise, n_matched = len(pipeline.resp.cell_ids), len(pipeline.analysis_chunk.cell_ids), len(pipeline.match_dict)
print(f'{grating_datafile}: {n_protocol} cells | {ANALYSIS_CHUNK}: {n_noise} cells | matched at {CORR_CUTOFF}: {n_matched} '
      f'({100 * n_matched / min(n_protocol, n_noise):.0f}% of the smaller run)')

corrs = np.array(list(pipeline.corr_dict.values()))
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(corrs, bins=30, color='0.5')
ax.axvline(CORR_CUTOFF, color='#e0201b', ls='--', label=f'cutoff {CORR_CUTOFF}')
ax.set_xlabel('EI correlation of accepted matches'); ax.set_ylabel('# cells'); ax.legend()
plt.show()

**How much does the cutoff matter here?** Re-run the matcher at 0.80 on the same two runs. The difference is the pairs that sit between 0.80 and 0.85: the ones a looser cutoff would let in.

In [ ]:
loose_matches, loose_corrs = ra.cluster_match(pipeline.analysis_chunk, pipeline.resp, corr_cutoff=0.80, verbose=False)
print(f'0.80: {len(loose_matches)} matches | 0.85: {n_matched} matches | extra at 0.80: {len(loose_matches) - n_matched}')

**Check it: the different-retina negative control.** Match this experiment's noise chunk against a chunk from a *different* retina. No cell can truly match, so every accepted pair is a false match. It should be ~0; if it isn't, the cutoff isn't safe for this kind of run. Run this whenever you join a new kind of run. For example, flash runs' EIs gave a few false matches even at 0.85 when we tried joining them to gratings.

In [ ]:
OTHER_EXP, OTHER_CHUNK = '20260506A', 'data006'   # any chunk from a different retina in your database

with ra.scrollable_prints():
    other_chunk = ra.AnalysisChunk(OTHER_EXP, OTHER_CHUNK, b_load_spatial_maps=False)
false_matches, false_corrs = ra.cluster_match(pipeline.analysis_chunk, other_chunk, corr_cutoff=CORR_CUTOFF, verbose=False)
print(f'{EXP_NAME} {ANALYSIS_CHUNK} vs {OTHER_EXP} {OTHER_CHUNK}: {len(false_matches)} matches at {CORR_CUTOFF} (should be ~0)')

## 3. Responses and normalization

### What number is "the response"?

- **Mean rate** (spikes in the stimulus window ÷ its duration, Hz) works for any stimulus. It's the only option for **steps** (flashes, spots), which have no rhythm to lock to.
- **F1** is the size of the response at the grating's own temporal frequency, i.e. how strongly the cell's firing follows each light/dark cycle. For **drifting gratings** F1 catches responses that mean rate misses. A cell that fires to every bright bar and goes silent for every dark bar can have almost the same *mean* rate as at rest, but a large F1.

**Decision:** gratings → F1; flashes/spots → mean rate. `ra.build_trial_response_table` computes both for gratings, so you can compare.

### Baseline (noise subtraction)

Two conventions exist (`baseline_condition_key`):
- **Pre-stimulus window** (the default): each trial's own spontaneous rate just before the stimulus.
- **0% contrast trials** (`baseline_condition_key='contrast'`): each cell's response to the blank condition.

**Decision:** 0% contrast. The pre-stimulus window is short (often 250 ms), and F1 measured over a short window has a noise floor well above zero even with no real response. Subtracting it drives F1 spuriously negative. Our own MATLAB CRF scripts plot raw F1.

In [ ]:
typing_chunk = pipeline.analysis_chunk   # reuse the reference chunk already loaded above
with ra.scrollable_prints():
    df_trials, spike_times_by_cell, df_epochs, _, ndf = ra.load_contrast_section(
        EXP_NAME, contrast_search, GRATING_PROTOCOL, condition_keys=['contrast'],
        analysis_chunk_name=ANALYSIS_CHUNK, corr_cutoff=CORR_CUTOFF, typing_chunk=typing_chunk,
        baseline_condition_key='contrast', baseline_condition_value=0.0,
    )
display(df_trials[['cell_id', 'cell_type', 'contrast', 'mean_rate', 'f1', 'mean_rate_noise_sub']].head())

### Per-cell vs population normalization

Each figure below has two columns:
- **Left, not normalized (Hz):** answers "how strongly does this type respond?" It keeps gain differences, but high-rate cells dominate the average.
- **Right, per-cell normalized (each cell's own max = 1):** answers "what's the *shape* of the curve?" (where it rises and where it saturates). Every cell counts equally, but differences in overall gain are removed by construction.

**Decision:** compare curve shape (and c50 from the Naka–Rushton fit) on per-cell-normalized curves, and compare response strength on raw Hz. Say which one a figure shows.

In [ ]:
crf_f1 = ra.plot_crf(df_trials, condition_key='contrast', response_col='f1', raw_response_col='f1',
                     title=f'{EXP_NAME} gratings: F1 vs contrast (NDF {ndf})', show_noise_sub=False, even_spacing=True)
crf_rate = ra.plot_crf(df_trials, condition_key='contrast', response_col='mean_rate', raw_response_col='mean_rate',
                       title=f'{EXP_NAME} gratings: mean rate vs contrast (NDF {ndf})', show_noise_sub=False, even_spacing=True)

**Naka–Rushton fits** are done on the per-cell-normalized F1 curve. A fit is only trustworthy if the cell actually saturates within the tested contrasts. If the curve is still rising at the highest contrast, c50 is extrapolated beyond the data. The contrast demo marks those cells as not well constrained and leaves them out of the c50 summary.

In [ ]:
curves = df_trials.groupby(['cell_id', 'contrast'])['f1'].mean().reset_index()
fits = []
for cell_id, grp in curves.groupby('cell_id'):
    grp = grp.sort_values('contrast')
    peak = grp['f1'].max()
    if not np.isfinite(peak) or peak <= 0:
        continue
    try:
        fit = ra.fit_naka_rushton(grp['contrast'].values, grp['f1'].values / peak)
        fit['cell_id'] = cell_id
        fits.append(fit)
    except RuntimeError:
        pass
df_fits = pd.DataFrame(fits)
print(f'{len(df_fits)} cells fit')
display(df_fits.describe().round(3))

## 4. Receptive fields and direction/orientation tuning

### RF shape: three views of the same STAs
- **Gaussian fit:** Vision fits a 2-D Gaussian to each STA (`SigmaX`, `SigmaY`, `Theta` in the `.params` file). Fast and compact, but only as good as the fit. Note: the ellipses are drawn with `std_scaling=1.6` passed as the ellipse's *width*, so each is drawn at a radius of 0.8 SD.
- **Contours:** traced straight from each cell's raw STA (no fit) around its center. With `contour_level='auto'`, the outline level is picked separately for each cell type as the level where that type's outlines tile best. The score is the uniformity index (Gauthier et al. 2009): the fraction of the mosaic covered by exactly one cell. A real mosaic tiles, so this avoids both piled-up blobs and gaps.
- **Portraits + timecourses:** each cell's raw STA pixels, with its timecourse (red) over its type's mean (blue). This is QC: a cell whose timecourse doesn't match its type (`r` below 0.8, red frame) may be misclassified or bad.

In [ ]:
rfs_gauss = pipeline.analysis_chunk.plot_rfs(minimum_n=3, b_zoom=True, typing_file=TYPING_FILE, title='Gaussian fits')
rfs_contour = pipeline.analysis_chunk.plot_rfs(minimum_n=3, b_zoom=True, typing_file=TYPING_FILE,
                                               use_contours=True, contour_level='auto', title='Contours (auto level)')

In [ ]:
PORTRAIT_TYPE = None   # e.g. 'off/brisk sustained'; None = every type with >= 3 cells
portraits = pipeline.analysis_chunk.plot_rf_portraits(cell_types=PORTRAIT_TYPE, minimum_n=3, typing_file=TYPING_FILE,
                                                      with_timecourses=True, tc_flag_r=0.8)

### RF size

`get_rf_sizes_from_sta` gives two diameters per cell:
- **`rf_diameter_halfmax_um` (use this one):** the stixels around the peak that are at least 50% of the cell's own peak value (center only), as the diameter of a circle with that area. The edge is tied to the cell's own peak, so a cleaner or noisier STA doesn't systematically move it. (A rule like "5 noise SDs above baseline" does: a cleaner STA pushes more of the RF's fading edge over the line, so the same cell measures bigger.)
- **`rf_diameter_gauss_um`:** from Vision's Gaussian fit, with the MATLAB convention (`plot_rf_fit.m` draws a 1-SD radius, so diameter = 2 × √(σx·σy)). Use it to compare with MATLAB results and papers that report fits.

**Which comparisons:**
- *Across types* in one experiment: do transient and sustained, or ON and OFF, differ?
- *Within a type*: the spread and outliers. An outlying cell is worth a look in the portraits above.
- *The same type across experiments or genotypes*: this belongs in the genotype comparison notebook. Always use the same method and say which.

**Precision:** RFs here are only 6–8 stixels across, so one stixel at the edge is ~15% of the diameter. Trust type medians over single cells. `sta_snr` is included so very noisy STAs can be dropped.

The two methods don't give the same number. For a perfectly Gaussian RF, half-max would be 1.18× the fit diameter; on 20251016A and 20260506A it is 1.35–1.5× for every type, so the raw STA centers are wider than Vision's σ implies. It's consistent across types, so comparisons *within* either method hold, but don't mix the two.

In [ ]:
df_rf_sizes = pipeline.analysis_chunk.get_rf_sizes_from_sta(minimum_n=3, typing_file=TYPING_FILE)
display(df_rf_sizes.groupby('cell_type')[['rf_diameter_halfmax_um', 'rf_diameter_gauss_um']].median().round(1))
fig_rf_sizes = ra.plot_rf_sizes(df_rf_sizes, title=f'{EXP_NAME} RF size by cell type')

### Direction and orientation selectivity (DS/OS)

For each cell:
1. Pick the one spatial/temporal-frequency condition with the highest peak response.
2. Compute DSI and OSI from its tuning curve with `ra.compute_dsi_osi` (vector sum; OSI uses doubled angles, so it's blind to which way along an axis the grating moves).
3. Classify: **DS** if DSI > 0.3; otherwise **OS** if OSI > 0.3; and in both cases only if the peak rate is > 2 Hz. DS takes precedence, so a cell is never both.

**Decision to be aware of: mean rate vs F1.** The DS/OS demo (and the lab's `DS-OS.ipynb`) uses mean rate. Our standalone DS/OS master table uses F1, because mean rate undercounts OS cells: on 20251016A, 2.9% OS with mean rate vs 14.3% with F1. The cell below computes both so you can see the difference on your data.

**Also know:** older runs used 6 directions (20 reps) and newer ones 12 (5 reps), and 6 directions inflates the index. The spatial period changes DS yield. Preferred directions are in screen coordinates: fine within one retina, but don't pool them across retinas, since each retina sits at its own rotation on the rig.

In [ ]:
DSOS_PROTOCOL = 'manookinlab.protocols.GratingDSOS'
DSI_T, OSI_T, MIN_HZ = 0.3, 0.3, 2.0

with ra.scrollable_prints():
    df_dsos, _, _, dsos_datafile, _ = ra.load_contrast_section(
        EXP_NAME, dsos_search, DSOS_PROTOCOL, condition_keys=['orientation', 'spatialFrequency', 'temporalFrequency'],
        analysis_chunk_name=ANALYSIS_CHUNK, corr_cutoff=CORR_CUTOFF, typing_chunk=typing_chunk,
    )

def classify(metric):
    tuning = df_dsos.groupby(['cell_id', 'spatialFrequency', 'temporalFrequency', 'orientation'])[[metric, 'mean_rate']].mean().reset_index()
    rows = []
    for cell_id, g in tuning.groupby('cell_id'):
        best = max(g.groupby(['spatialFrequency', 'temporalFrequency']), key=lambda kv: kv[1]['mean_rate'].max())[1]
        res = ra.compute_dsi_osi(best['orientation'].values, best[metric].clip(lower=0).values)
        res.update(cell_id=cell_id, peak_rate=best['mean_rate'].max())
        rows.append(res)
    d = pd.DataFrame(rows)
    d['class'] = np.where((d.dsi > DSI_T) & (d.peak_rate > MIN_HZ), 'DS',
                 np.where((d.osi > OSI_T) & (d.peak_rate > MIN_HZ), 'OS', 'none'))
    return d

for metric in ['mean_rate', 'f1']:
    d = classify(metric)
    counts = d['class'].value_counts()
    print(f'{metric:>9}: {len(d)} cells | DS {counts.get("DS", 0)} ({100 * counts.get("DS", 0) / len(d):.1f}%) '
          f'| OS {counts.get("OS", 0)} ({100 * counts.get("OS", 0) / len(d):.1f}%)')

## 5. Saving figures and data

- `ra.save_figure(fig, 'folder/name')` writes `.svg`, `.pdf` and `.png`. Text stays editable in Illustrator, and very dense plots (rasters, big scatters) are flattened to an embedded image so the file still opens quickly.
- `ra.export_figure_data(fig, 'folder/name')` writes `name.csv` with the numbers actually plotted, one column per series (e.g. `on/brisk sustained | series1 | y`). That's the source-data format journals ask for, and it reads straight into Igor or MATLAB (`readtable`).

In [ ]:
OUT = f'figures/{EXP_NAME}'
ra.save_figure(crf_f1, f'{OUT}/crf_f1')
ra.export_figure_data(crf_f1, f'{OUT}/crf_f1')

## Where to go next
- `1_database_demo`: search the database and plot mosaics across experiments.
- `2_data_quality_demo`: QC a chunk (RF mosaics, contours, portraits + timecourses, RF size, EI footprints across NDFs).
- `3_contrast_grating_demo`, `5_flash_demo`, `6_spot_demo`: contrast-response curves per stimulus, across NDFs.
- `4_grating_dsos_demo`: DS/OS for one experiment with polar/raster plots.
- `7_correlated_spiking_demo`: cross-correlations across light levels.
- `8_genotype_comparison_demo`: compare genotypes across experiments.